# Prepare NEB Path Materials

Build an **ordered materials set** for Nudged Elastic Band: start from one structure, transform it into a final image (example: move an atom), then save **first** and **last** into a set for [`neb.ipynb`](neb.ipynb).

## Usage

1. Set material / set name in cell 1.2 and optional displacement params in cell 1.3.
1. Edit the transformation cell to apply any change you need (`set_coordinates` example provided).
1. Run all cells.
1. Copy the printed `MATERIAL_SET` name into `neb.ipynb` (and set `N_IMAGES` there if you only saved first+last).

## Summary

1. Install packages and set parameters.
1. Authenticate and select account.
1. Load the starting material.
1. Clone as initial; transform a copy into the final image.
1. Save both materials and place them in an ordered materials set.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
# Auth / organization
ORGANIZATION_NAME = None

# Starting material (uploads folder or Standata name match)
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"

# Ordered set name — same string as MATERIAL_SET in neb.ipynb
MATERIAL_SET_NAME = "NEB Silicon"


### 1.3. Set example transformation parameters
Used by the example `set_coordinates` cell below — replace with your own logic as needed.

In [ ]:
# Example only: displace one site in crystal coordinates
ATOM_INDEX = 0
DISPLACEMENT = [0.0, 0.0, 0.1]

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

## 3. Build initial and final images
### 3.1. Load starting material

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

source_material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME)
)
visualize(source_material)

### 3.2. Clone as initial image

In [ ]:
initial_material = source_material.clone()
initial_material.name = f"{source_material.name}-initial"
print(f"Initial: {initial_material.name}")
visualize(initial_material)

### 3.3. Transform into the final image

Example uses `set_coordinates` to displace one atom. **Edit this cell** for any transformation you need (defects, swaps, custom coordinates, etc.).

In [ ]:
final_material = initial_material.clone()
coordinates = [list(coordinate) for coordinate in final_material.coordinates_array]
for axis_index, delta in enumerate(DISPLACEMENT):
    coordinates[ATOM_INDEX][axis_index] += delta
final_material.set_coordinates(coordinates)
final_material.name = f"{source_material.name}-final"
print(f"Final: {final_material.name}")
print(f"Atom {ATOM_INDEX}: {initial_material.coordinates_array[ATOM_INDEX]} → {final_material.coordinates_array[ATOM_INDEX]}")
visualize([initial_material, final_material])

## 4. Save to an ordered materials set
### 4.1. Create or reuse materials on the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_initial = get_or_create_material(client, initial_material, ACCOUNT_ID)
saved_final = get_or_create_material(client, final_material, ACCOUNT_ID)
print(f"Initial ID: {saved_initial['_id']}")
print(f"Final ID: {saved_final['_id']}")

### 4.2. Create ordered set and move first → last

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import create_ordered_materials_set

materials_set = create_ordered_materials_set(
    client,
    ACCOUNT_ID,
    MATERIAL_SET_NAME,
    [saved_initial, saved_final],
)
print(f"MATERIAL_SET = {materials_set['name']!r}")
